# Preprocessing Validation & Progression Visualization

This notebook validates the full windowing pipeline and visualizes how brain activity
evolves across the 30 minutes leading up to a seizure.

**Goals:**
1. Confirm `.npy` sequences have the correct shape and content
2. Inspect the metadata index
3. Visualize the progression of brain maps across time for one seizure event
4. Compare pre-ictal vs interictal brain map patterns side by side
5. Verify zero-padding behaviour for short sequences
6. Check class balance across the full dataset

**Run this notebook after `build_dataset.py` has completed successfully.**

---
## 1. Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import mne

# Make src/ importable from the notebook
sys.path.insert(0, '../src')

mne.set_log_level('WARNING')

# Paths
SEQUENCES_DIR = '../data/processed/sequences'
METADATA_PATH = '../data/processed/metadata.csv'
CHBMIT_ROOT   = '../data/raw/chb-mit'

# Band names for labelling plots
BAND_NAMES = ['Delta\n0.5–4 Hz', 'Theta\n4–8 Hz', 'Alpha\n8–13 Hz',
              'Beta\n13–30 Hz', 'Gamma\n30–40 Hz']

print('Imports OK.')
print(f'Sequences dir : {os.path.abspath(SEQUENCES_DIR)}')
print(f'Metadata file : {os.path.abspath(METADATA_PATH)}')

---
## 2. Metadata Inspection

Load and inspect `metadata.csv` — the index of every sequence in the dataset.

In [ ]:
meta = pd.read_csv(METADATA_PATH)

print(f'Total sequences : {len(meta)}')
print(f'Columns         : {list(meta.columns)}')
print()
meta.head(10)

In [ ]:
# --- Class balance overview ---
counts = meta['type'].value_counts()
print('Class distribution:')
print(counts.to_string())
print(f'\nRatio preictal:interictal = 1 : {counts.get("interictal", 0) / max(counts.get("preictal", 1), 1):.2f}')

In [ ]:
# --- Per-patient sequence counts ---
per_patient = meta.groupby(['patient_id', 'type']).size().unstack(fill_value=0)
per_patient['total'] = per_patient.sum(axis=1)
print('Sequences per patient:')
print(per_patient.to_string())

# Plot
fig, ax = plt.subplots(figsize=(14, 4))
per_patient[['preictal', 'interictal']].plot(
    kind='bar', ax=ax,
    color=['#e05c5c', '#5c8ae0'],
    edgecolor='white', width=0.7
)
ax.set_title('Sequences per patient', fontsize=12)
ax.set_xlabel('Patient')
ax.set_ylabel('Count')
ax.legend(['Pre-ictal', 'Interictal'])
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../experiments/results/dataset_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/dataset_balance.png')

---
## 3. Single Sequence Shape Verification

Load one pre-ictal `.npy` file and confirm shape, dtype, and value ranges.

In [ ]:
# Pick the first pre-ictal sequence in the metadata
first_preictal = meta[meta['type'] == 'preictal'].iloc[0]
seq_path = os.path.join(SEQUENCES_DIR, first_preictal['filename'])

seq = np.load(seq_path)

nz = (seq.sum(axis=(1,2)) != 0).sum()
print(f'Real frames: {nz} / 360  ({nz * 5 / 60:.1f} minutes of actual data)')

print(f'File          : {first_preictal["filename"]}')
print(f'Patient       : {first_preictal["patient_id"]}')
print(f'Source EDF    : {first_preictal["source_edf"]}')
print(f'Seizure onset : {first_preictal["anchor_sec"]}s')
print()
print(f'Array shape   : {seq.shape}   ← (n_frames, n_bands, n_channels)')
print(f'dtype         : {seq.dtype}')
print(f'Min value     : {seq.min():.4e}')
print(f'Max value     : {seq.max():.4e}')
print(f'Zero frames   : {(seq.sum(axis=(1,2)) == 0).sum()}  (zero-padded frames at start)')
print(f'Non-zero frames: {(seq.sum(axis=(1,2)) != 0).sum()}')

In [ ]:
# --- Visualise zero-padding pattern ---
# Plot the total power per frame across time to see where padding ends
frame_energy = seq.sum(axis=(1, 2))  # one value per frame

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(frame_energy, color='steelblue', linewidth=0.8)
ax.set_title(f'Total band power per frame — {first_preictal["filename"]}\n'
             f'(flat zero region at start = zero-padding)', fontsize=11)
ax.set_xlabel('Frame index (0 = 30 min before seizure, 359 = seizure onset)')
ax.set_ylabel('Total power (all bands + channels)')
ax.axvline(x=(frame_energy != 0).argmax(), color='red', linestyle='--',
           label='Padding ends here')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Brain Map Progression Visualization

This is the core visualization: a sequence of topographic brain maps showing how
brain activity evolves across the 30 minutes before seizure onset.

We sample **6 evenly spaced frames** from the non-padded portion of the sequence:
- Frame 1: ~30 min before seizure  
- Frame 6: ~5 min before seizure  

Each row is one frequency band. Each column is one time point.

In [ ]:
# --- Build electrode positions from CHB-MIT channel names ---
# CHB-MIT uses bipolar pairs (e.g. 'FP1-F7') — extract first electrode
# of each pair and look up its 2D position in the standard 10-20 montage

# Load channel names from the source EDF
edf_path = os.path.join(CHBMIT_ROOT,
                        first_preictal['patient_id'],
                        first_preictal['source_edf'])

raw_ref = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
ch_names = raw_ref.ch_names
print(f'Channels in recording: {len(ch_names)}')
print(ch_names)

# Build (x, y) position array from montage
montage     = mne.channels.make_standard_montage('standard_1020')
montage_pos = montage.get_positions()['ch_pos']

positions  = []
valid_idx  = []
valid_names = []

for i, ch in enumerate(ch_names):
    first_el = ch.split('-')[0].strip().upper()
    if first_el in montage_pos:
        xyz = montage_pos[first_el]
        positions.append(xyz[:2])   # X and Y only
        valid_idx.append(i)
        valid_names.append(first_el)

positions = np.array(positions)
print(f'\nChannels with valid scalp positions: {len(valid_idx)} / {len(ch_names)}')
print(f'Valid channels: {valid_names}')

In [ ]:
# --- Select 6 evenly spaced frames from the non-padded region ---
non_zero_mask   = seq.sum(axis=(1, 2)) != 0
non_zero_frames = np.where(non_zero_mask)[0]  # indices of real (non-padded) frames

n_snapshots = 6
snapshot_indices = np.linspace(
    non_zero_frames[0],
    non_zero_frames[-1],
    n_snapshots,
    dtype=int
)

# Convert frame indices to minutes before seizure
# Frame 359 = seizure onset (t=0), frame 0 = 30 min before
TOTAL_FRAMES  = seq.shape[0]   # 360
WINDOW_SECS   = 5

def frame_to_minutes_before(frame_idx):
    seconds_before = (TOTAL_FRAMES - 1 - frame_idx) * WINDOW_SECS
    return seconds_before / 60

time_labels = [f't−{frame_to_minutes_before(i):.0f} min' for i in snapshot_indices]
print('Snapshot time points:', time_labels)

# --- Build the figure: 5 rows (bands) × 6 columns (time points) ---
n_bands     = 5
fig = plt.figure(figsize=(20, 16))
fig.suptitle(
    f'Pre-ictal Brain Map Progression — {first_preictal["patient_id"].upper()}\n'
    f'30 minutes → seizure onset  |  Each row = one frequency band  |  '
    f'Red = high power, Blue = low power',
    fontsize=13, y=0.98
)

gs = gridspec.GridSpec(
    n_bands, n_snapshots,
    figure=fig,
    hspace=0.05,
    wspace=0.05
)

band_labels = ['Delta\n0.5–4 Hz', 'Theta\n4–8 Hz', 'Alpha\n8–13 Hz',
               'Beta\n13–30 Hz', 'Gamma\n30–40 Hz']

for band_idx in range(n_bands):
    for col_idx, frame_idx in enumerate(snapshot_indices):
        ax = fig.add_subplot(gs[band_idx, col_idx])

        # Extract band power for valid channels at this frame
        frame_data   = seq[frame_idx, band_idx, :]      # (n_channels,)
        power_valid  = frame_data[valid_idx]             # keep only positioned channels

        # Normalize per-band across all time points for consistent color scale
        # band_all_frames = seq[non_zero_frames, band_idx, :][:, valid_idx]
        # global_min = band_all_frames.min()
        # global_max = band_all_frames.max()
        # power_norm = (power_valid - global_min) / (global_max - global_min + 1e-12)
        # AFTER — per-frame normalization
        power_norm = (power_valid - power_valid.min()) / (power_valid.max() - power_valid.min() + 1e-12)

        mne.viz.plot_topomap(
            power_norm,
            positions,
            axes=ax,
            show=False,
            cmap='RdYlBu_r',
            vlim=(0, 1),
            contours=4,
            sphere=0.07
        )

        # Column headers (time labels) on top row only
        if band_idx == 0:
            ax.set_title(time_labels[col_idx], fontsize=9, pad=6,
                         color='#cc0000' if col_idx == n_snapshots - 1 else 'black',
                         fontweight='bold' if col_idx == n_snapshots - 1 else 'normal')

        # Row labels (band names) on leftmost column only
        if col_idx == 0:
            ax.set_ylabel(band_labels[band_idx], fontsize=9, labelpad=4)

plt.savefig('../experiments/results/progression_preictal.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/results/progression_preictal.png')
print('\nThis figure is a candidate for Figure 1 in your paper.')

---
## 5. Pre-ictal vs Interictal Side-by-Side Comparison

Compare the brain map at the **last frame before seizure** (most pre-ictal)
against an **interictal frame** from the same patient.

This is the clearest visual evidence that your pipeline captured
a real neurological difference — not noise.

In [ ]:
# --- Load a matching interictal sequence from the same patient ---
patient_id       = first_preictal['patient_id']
inter_candidates = meta[(meta['patient_id'] == patient_id) & (meta['type'] == 'interictal')]

if len(inter_candidates) == 0:
    print(f'No interictal sequences found for {patient_id}. Try a different patient.')
else:
    inter_row  = inter_candidates.iloc[0]
    inter_seq  = np.load(os.path.join(SEQUENCES_DIR, inter_row['filename']))

    # Use last non-zero frame from each sequence
    preictal_frame  = seq[non_zero_frames[-1]]     # shape: (5, n_channels)
    inter_nz        = np.where(inter_seq.sum(axis=(1,2)) != 0)[0]
    interictal_frame = inter_seq[inter_nz[-1]]     # shape: (5, n_channels)

    fig, axes = plt.subplots(2, n_bands, figsize=(20, 8))
    fig.suptitle(
        f'Pre-ictal vs Interictal Brain Maps — {patient_id.upper()}\n'
        f'Top row: last frame before seizure  |  Bottom row: interictal (no seizure)',
        fontsize=12, y=1.01
    )

    row_labels = ['Pre-ictal\n(last frame)', 'Interictal\n(resting)']
    frames     = [preictal_frame, interictal_frame]

    for row_idx, (frame, row_label) in enumerate(zip(frames, row_labels)):
        for band_idx in range(n_bands):
            ax = axes[row_idx, band_idx]

            power_valid = frame[band_idx, valid_idx]
            power_norm  = (power_valid - power_valid.min()) / \
                          (power_valid.max() - power_valid.min() + 1e-12)

            mne.viz.plot_topomap(
                power_norm,
                positions,
                axes=ax,
                show=False,
                cmap='RdYlBu_r',
                vlim=(0, 1),
                contours=4,
                sphere=0.07
            )

            if row_idx == 0:
                ax.set_title(BAND_NAMES[band_idx], fontsize=10, pad=8)
            if band_idx == 0:
                ax.set_ylabel(row_label, fontsize=10)

    plt.tight_layout()
    plt.savefig('../experiments/results/preictal_vs_interictal.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: experiments/results/preictal_vs_interictal.png')
    print('\nThis figure is a candidate for Figure 2 in your paper.')

---
## 6. Cross-Patient Consistency Check

Check whether the pre-ictal pattern (elevated delta/theta in temporal regions)
is consistent across multiple patients, or patient-specific.

Plot the **average last-frame brain map** across all patients for delta band only.

In [ ]:
import warnings
# Collect last pre-ictal frame delta power across all patients
all_last_frames = []
patient_ids_used = []

preictal_meta = meta[meta['type'] == 'preictal']

for _, row in preictal_meta.iterrows():
    s = np.load(os.path.join(SEQUENCES_DIR, row['filename']))
    nz = np.where(s.sum(axis=(1, 2)) != 0)[0]
    if len(nz) == 0:
        continue

    n_ch = s.shape[2]  # actual channel count for this sequence

    # Rebuild valid_idx for this sequence's channel count
    # Load channel names from the source EDF for this patient
    edf_p = os.path.join(CHBMIT_ROOT, row['patient_id'], row['source_edf'])
    if not os.path.exists(edf_p):
        continue

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            raw_p   = mne.io.read_raw_edf(edf_p, preload=False, verbose=False)
        chs_p   = raw_p.ch_names[:n_ch]  # match sequence channel count
    except Exception:
        continue

    local_valid_idx = []
    for i, ch in enumerate(chs_p):
        first_el = ch.split('-')[0].strip().upper()
        if first_el in montage_pos:
            local_valid_idx.append(i)

    if len(local_valid_idx) == 0:
        continue

    last_frame = s[nz[-1], 0, local_valid_idx]   # band 0 = delta

    # Normalize
    r = last_frame.max() - last_frame.min()
    if r > 0:
        last_frame = (last_frame - last_frame.min()) / r

    # We can only average channels that exist in both this sequence
    # and the reference positions array — use the minimum overlap
    n_valid = min(len(last_frame), len(positions))
    all_last_frames.append(last_frame[:n_valid])
    patient_ids_used.append(row['patient_id'])

# Pad to uniform length before averaging
max_len = max(len(f) for f in all_last_frames)
padded  = np.array([
    np.pad(f, (0, max_len - len(f))) for f in all_last_frames
])

mean_delta = np.mean(padded, axis=0)
std_delta  = np.std(padded, axis=0)

# Trim positions to match
plot_positions = positions[:max_len]

print(f'Averaged over {len(all_last_frames)} pre-ictal sequences from '
      f'{len(set(patient_ids_used))} patients')